# Credit Card Fraud Detection Pipeline
### Machine Learning Benchmark: Logistic Regression vs. Random Forest with SMOTE Imbalance Handling

**Author:** Jabulani Mayenda  
**Repository:** [github.com/jabulani-mayenda/fruad-detection](https://github.com/jabulani-mayenda/fruad-detection)  

---

## 1. Executive Summary & Problem Formulation
Financial transaction systems process billions of operations daily. Among these, fraudulent activities account for a tiny fraction (~0.17%), yet inflict catastrophic financial and reputational losses.

### Key Challenges:
1. **Extreme Class Imbalance:** Normal transactions outnumber fraud ~578 to 1. Traditional classifiers optimizing accuracy trivially achieve 99.8% by predicting zero fraud, missing all attacks.
2. **Asymmetric Error Costs:** False Negatives (allowing fraud) cause immediate capital loss; False Positives cause customer friction.
3. **Dimensionality & Privacy:** Features $V_1$ through $V_{28}$ are PCA-transformed to protect cardholder privacy, requiring algorithms that excel on latent spatial manifolds.

This study compares two algorithmic paradigms:
- **Logistic Regression** (Linear probabilistic model with Balanced Cost Weighting)
- **Random Forest Classifier** (Non-linear ensemble with bootstrap aggregation)


## 2. Environment Setup & Dependency Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
from imblearn.over_sampling import SMOTE

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
print('[+] All libraries successfully imported.')

## 3. Dataset Loading & Exploratory Data Analysis (EDA)

In [2]:
# Load the dataset
csv_path = 'creditcard.csv'
df = pd.read_csv(csv_path)

print(f'Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
display(df.head())

# Target Distribution Analysis
class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100

print(f'Class 0 (Legitimate): {class_counts[0]:,} ({100 - fraud_pct:.3f}%)')
print(f'Class 1 (Fraudulent): {class_counts[1]:,} ({fraud_pct:.3f}%)')
print(f'Imbalance Ratio: {class_counts[0] // class_counts[1]}:1')

### Visualizing Class Imbalance and Transaction Amounts

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class Imbalance Bar Plot
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], [class_counts[0], class_counts[1]], color=['#2563eb', '#dc2626'])
axes[0].set_yscale('log')
axes[0].set_title('Transaction Volume by Class (Log Scale)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Transactions (Log)')

# Amount Distribution Comparison
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1], palette=['#2563eb', '#dc2626'])
axes[1].set_yscale('log')
axes[1].set_title('Transaction Amount by Class ($)', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

## 4. Feature Preprocessing & Stratified Splitting
- PCA features $V_1$ to $V_{28}$ are already zero-centered with standard unit variance.
- `Time` and `Amount` must be standardized to prevent scale dominance.
- We perform an 80/20 Stratified Split to guarantee exact fraud representation across train and test partitions.

In [4]:
X = df.drop(columns=['Class'])
y = df['Class']

# Standardize Time and Amount
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Samples: {len(X_train):,} ({y_train.sum():,} fraud)')
print(f'Testing Samples:  {len(X_test):,} ({y_test.sum():,} fraud)')

## 5. Resampling with SMOTE (Synthetic Minority Over-sampling)
**Crucial Machine Learning Rule:** SMOTE must ONLY be applied to the **training partition** to prevent data leakage.

In [5]:
print(f'Before SMOTE - Class 0: {(y_train == 0).sum():,}, Class 1: {(y_train == 1).sum():,}')

smote = SMOTE(sampling_strategy=0.20, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f'After SMOTE  - Class 0: {(y_train_smote == 0).sum():,}, Class 1: {(y_train_smote == 1).sum():,}')

## 6. Model 1: Logistic Regression
Trained with class balancing on the resampled feature space.

In [6]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_smote, y_train_smote)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print('--- LOGISTIC REGRESSION CLASSIFICATION REPORT ---')
print(classification_report(y_test, lr_pred, digits=4))
print(f'ROC-AUC Score: {roc_auc_score(y_test, lr_prob):.4f}')
print(f'Average Precision (PR-AUC): {average_precision_score(y_test, lr_prob):.4f}')

## 7. Model 2: Random Forest Classifier
An ensemble of 100 decorrelated decision trees with max depth regularized at 14.

In [7]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=14,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print('--- RANDOM FOREST CLASSIFICATION REPORT ---')
print(classification_report(y_test, rf_pred, digits=4))
print(f'ROC-AUC Score: {roc_auc_score(y_test, rf_prob):.4f}')
print(f'Average Precision (PR-AUC): {average_precision_score(y_test, rf_prob):.4f}')

## 8. Comparative Evaluation & Diagnostic Visualizations

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrices
cm_lr = confusion_matrix(y_test, lr_pred)
cm_rf = confusion_matrix(y_test, rf_pred)

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Reds', ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

In [9]:
# ROC and Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)

axes[0].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_prob):.4f})', color='#2563eb', lw=2)
axes[0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_prob):.4f})', color='#dc2626', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('ROC Curves', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# PR Curves
prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_prob)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_prob)

axes[1].plot(rec_lr, prec_lr, label=f'Logistic Regression (AP = {average_precision_score(y_test, lr_prob):.4f})', color='#2563eb', lw=2)
axes[1].plot(rec_rf, prec_rf, label=f'Random Forest (AP = {average_precision_score(y_test, rf_prob):.4f})', color='#dc2626', lw=2)
axes[1].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Top Fraud Predictive Indicators (Feature Importance)
Visualizing which latent PCA features provide the strongest signal for fraud identification.

In [10]:
feature_names = list(X.columns)
importances = rf.feature_importances_
top_indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.barh([feature_names[i] for i in top_indices][::-1], [importances[i] for i in top_indices][::-1], color='#dc2626')
plt.title('Top 10 Feature Importances (Random Forest Gini Impurity Reduction)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 10. Real-time Inference Function
Function that scores a single transaction dictionary and issues automated fraud alerts.

In [11]:
def detect_fraud(transaction_dict):
    df_single = pd.DataFrame([transaction_dict])
    df_single[['Time', 'Amount']] = scaler.transform(df_single[['Time', 'Amount']])
    
    rf_prob = rf.predict_proba(df_single)[0, 1]
    lr_prob = lr.predict_proba(df_single)[0, 1]
    consensus_prob = (rf_prob * 0.7) + (lr_prob * 0.3)
    
    decision = 'FRAUD' if consensus_prob >= 0.50 else 'LEGITIMATE'
    print(f'Decision:       {decision}')
    print(f'Consensus Prob: {consensus_prob*100:.2f}%')
    print(f'RF Prob:        {rf_prob*100:.2f}% | LR Prob: {lr_prob*100:.2f}%')
    return decision, consensus_prob

# Test on first test transaction
test_tx = X.iloc[y_test.index[0]].to_dict()
actual = y_test.iloc[0]
print(f'Ground Truth Label: {"FRAUD" if actual == 1 else "LEGITIMATE"}')
detect_fraud(test_tx);